# Modeling Human Activity States Using Hidden Markov Models
This notebook loads the collected Sensor Logger data, extracts features, trains a Gaussian Hidden Markov Model using Baum-Welch, decodes unseen activity states with Viterbi, and evaluates the model.

In [ ]:
import os, glob, shutil, zipfile, json, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score

PROJ=Path('..').resolve()
for p in ['data/raw','data/unseen','data/processed','figures']:
    (PROJ/p).mkdir(parents=True, exist_ok=True)

# Data is already placed inside data/raw and data/unseen.
zip_map={
 'standing':'standing-2026-07-01_07-32-53.zip', 'standing_test':'standing_test-2026-07-01_07-33-13.zip',
 'walking':'walking-2026-07-01_07-36-31.zip','walking_test':'walking_test-2026-07-01_07-36-44.zip',
 'jumping':'jumping-2026-07-01_07-34-30.zip','jumping_test':'jumping_test-2026-07-01_07-34-55.zip',
 'still':'still-2026-07-01_07-31-10.zip','still_test':'still_test-2026-07-01_07-32-03.zip'}
# Raw Sensor Logger folders are already included in the repository.

activities=['standing','walking','jumping','still']
state_names=activities
state_to_idx={s:i for i,s in enumerate(state_names)}

def sampling_rate(csv_path):
    df=pd.read_csv(csv_path)
    t=df['seconds_elapsed'].to_numpy()
    return 1/np.median(np.diff(t))

rates=[]
for split in ['raw','unseen']:
    for act in activities:
        path=PROJ/f'data/{split}/{act}{"_test" if split=="unseen" else ""}'/'Accelerometer.csv'
        rates.append({'split':split,'activity':act,'sensor':'Accelerometer','sampling_rate_hz':sampling_rate(path)})
        pathg=path.parent/'Gyroscope.csv'
        rates.append({'split':split,'activity':act,'sensor':'Gyroscope','sampling_rate_hz':sampling_rate(pathg)})
rates_df=pd.DataFrame(rates)
rates_df.to_csv(PROJ/'data/processed/sampling_rates.csv', index=False)
TARGET_FS=50.0

def load_record(folder, activity):
    acc=pd.read_csv(folder/'Accelerometer.csv').rename(columns={'x':'acc_x','y':'acc_y','z':'acc_z'})[['seconds_elapsed','acc_x','acc_y','acc_z']]
    gyr=pd.read_csv(folder/'Gyroscope.csv').rename(columns={'x':'gyro_x','y':'gyro_y','z':'gyro_z'})[['seconds_elapsed','gyro_x','gyro_y','gyro_z']]
    t0=max(acc.seconds_elapsed.min(), gyr.seconds_elapsed.min())
    t1=min(acc.seconds_elapsed.max(), gyr.seconds_elapsed.max())
    grid=np.arange(t0,t1,1/TARGET_FS)
    out=pd.DataFrame({'seconds_elapsed':grid-grid[0]})
    for col in ['acc_x','acc_y','acc_z']:
        out[col]=np.interp(grid, acc.seconds_elapsed, acc[col])
    for col in ['gyro_x','gyro_y','gyro_z']:
        out[col]=np.interp(grid, gyr.seconds_elapsed, gyr[col])
    out['activity']=activity
    return out

def features_from_window(w, fs=TARGET_FS):
    feats={}
    cols=['acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z']
    for c in cols:
        v=w[c].to_numpy()
        feats[f'{c}_mean']=float(np.mean(v)); feats[f'{c}_std']=float(np.std(v)); feats[f'{c}_var']=float(np.var(v))
        yf=np.abs(rfft(v-np.mean(v)))
        freqs=rfftfreq(len(v), 1/fs)
        feats[f'{c}_dom_freq']=float(freqs[1:][np.argmax(yf[1:])] if len(yf)>1 else 0)
        feats[f'{c}_spectral_energy']=float(np.sum(yf**2)/len(yf))
    acc_mag=np.sqrt(w.acc_x**2+w.acc_y**2+w.acc_z**2).to_numpy()
    gyro_mag=np.sqrt(w.gyro_x**2+w.gyro_y**2+w.gyro_z**2).to_numpy()
    feats['acc_sma']=float((np.abs(w.acc_x).sum()+np.abs(w.acc_y).sum()+np.abs(w.acc_z).sum())/len(w))
    feats['gyro_sma']=float((np.abs(w.gyro_x).sum()+np.abs(w.gyro_y).sum()+np.abs(w.gyro_z).sum())/len(w))
    feats['acc_mag_mean']=float(np.mean(acc_mag)); feats['acc_mag_std']=float(np.std(acc_mag))
    feats['gyro_mag_mean']=float(np.mean(gyro_mag)); feats['gyro_mag_std']=float(np.std(gyro_mag))
    feats['acc_xy_corr']=float(np.corrcoef(w.acc_x,w.acc_y)[0,1]) if len(w)>2 else 0
    feats['acc_xz_corr']=float(np.corrcoef(w.acc_x,w.acc_z)[0,1]) if len(w)>2 else 0
    feats['acc_yz_corr']=float(np.corrcoef(w.acc_y,w.acc_z)[0,1]) if len(w)>2 else 0
    for k,v in list(feats.items()):
        if not np.isfinite(v): feats[k]=0.0
    return feats

def make_features(split):
    rows=[]; raw_all=[]
    for act in activities:
        folder=PROJ/f'data/{split}'/(act+'_test' if split=='unseen' else act)
        df=load_record(folder, act)
        raw_all.append(df.assign(recording=f'{act}_{split}'))
        win=int(TARGET_FS*1.0); step=int(TARGET_FS*0.5); n=0
        for start in range(0, len(df)-win+1, step):
            w=df.iloc[start:start+win]
            f=features_from_window(w)
            f.update({'activity':act,'state':state_to_idx[act],'start_time':float(w.seconds_elapsed.iloc[0]),'end_time':float(w.seconds_elapsed.iloc[-1]),'split':split,'recording':f'{act}_{split}'})
            rows.append(f); n+=1
    return pd.DataFrame(rows), pd.concat(raw_all, ignore_index=True)
train_feat, train_raw=make_features('raw')
test_feat, test_raw=make_features('unseen')
train_feat.to_csv(PROJ/'data/processed/train_features.csv', index=False)
test_feat.to_csv(PROJ/'data/processed/unseen_features.csv', index=False)
train_raw.to_csv(PROJ/'data/processed/resampled_training_data.csv', index=False)
test_raw.to_csv(PROJ/'data/processed/resampled_unseen_data.csv', index=False)

feature_cols=[c for c in train_feat.columns if c not in ['activity','state','start_time','end_time','split','recording']]
scaler=StandardScaler().fit(train_feat[feature_cols])
X_train=scaler.transform(train_feat[feature_cols]); y_train=train_feat.state.to_numpy()
X_test=scaler.transform(test_feat[feature_cols]); y_test=test_feat.state.to_numpy()

class GaussianHMM:
    def __init__(self, n_states, n_features, n_iter=15, random_state=0):
        self.n_states=n_states; self.n_features=n_features; self.n_iter=n_iter; self.eps=1e-6
    def _log_gauss(self,X):
        N=len(X); logp=np.zeros((N,self.n_states))
        for k in range(self.n_states):
            var=self.vars_[k]+self.eps
            logp[:,k]=-0.5*(np.sum(np.log(2*np.pi*var))+np.sum((X-self.means_[k])**2/var,axis=1))
        return logp
    def fit_supervised_init(self,X,y):
        K=self.n_states
        self.pi_=np.bincount([y[0]], minlength=K).astype(float)+0.1; self.pi_/=self.pi_.sum()
        A=np.ones((K,K))*0.05
        for a,b in zip(y[:-1],y[1:]): A[a,b]+=1
        self.A_=A/A.sum(axis=1,keepdims=True)
        self.means_=np.vstack([X[y==k].mean(axis=0) if np.any(y==k) else X.mean(axis=0) for k in range(K)])
        self.vars_=np.vstack([X[y==k].var(axis=0)+0.1 if np.any(y==k) else X.var(axis=0)+0.1 for k in range(K)])
        # Baum-Welch EM refinement on concatenated labelled sequence treated as observations only
        for _ in range(self.n_iter):
            logB=self._log_gauss(X); N=len(X)
            la=np.zeros((N,K)); lb=np.zeros((N,K));
            la[0]=np.log(self.pi_)+logB[0]
            for t in range(1,N): la[t]=logB[t]+logsumexp(la[t-1][:,None]+np.log(self.A_),axis=0)
            lb[-1]=0
            for t in range(N-2,-1,-1): lb[t]=logsumexp(np.log(self.A_)+logB[t+1]+lb[t+1],axis=1)
            ll=logsumexp(la[-1])
            gamma=np.exp(la+lb-ll)
            xi=np.zeros((N-1,K,K))
            for t in range(N-1): xi[t]=np.exp(la[t][:,None]+np.log(self.A_)+logB[t+1]+lb[t+1]-ll)
            # anchor transitions/emissions mildly with supervised labels to prevent state switching
            supervised_gamma=np.eye(K)[y]
            gamma=0.7*supervised_gamma+0.3*gamma
            self.pi_=gamma[0]/gamma[0].sum()
            self.A_=xi.sum(axis=0)+0.1
            self.A_=self.A_/self.A_.sum(axis=1,keepdims=True)
            weights=gamma.sum(axis=0)+self.eps
            self.means_=(gamma.T@X)/weights[:,None]
            for k in range(K): self.vars_[k]=(gamma[:,k][:,None]*(X-self.means_[k])**2).sum(axis=0)/weights[k]+self.eps
        return self
    def viterbi(self,X):
        logB=self._log_gauss(X); K=self.n_states; N=len(X)
        delta=np.zeros((N,K)); psi=np.zeros((N,K),dtype=int)
        delta[0]=np.log(self.pi_)+logB[0]
        for t in range(1,N):
            scores=delta[t-1][:,None]+np.log(self.A_)
            psi[t]=np.argmax(scores,axis=0)
            delta[t]=np.max(scores,axis=0)+logB[t]
        path=np.zeros(N,dtype=int); path[-1]=np.argmax(delta[-1])
        for t in range(N-2,-1,-1): path[t]=psi[t+1,path[t+1]]
        return path

model=GaussianHMM(4, X_train.shape[1], n_iter=8).fit_supervised_init(X_train,y_train)
y_pred=model.viterbi(X_test)
# If Viterbi overly smooths across independent recordings, decode each recording separately and concatenate
preds=[]; truths=[]
for rec,grp in test_feat.groupby('recording', sort=False):
    idx=grp.index.to_numpy(); localX=scaler.transform(grp[feature_cols]); preds.extend(model.viterbi(localX)); truths.extend(grp.state.to_numpy())
y_pred=np.array(preds); y_test2=np.array(truths)
cm=confusion_matrix(y_test2,y_pred,labels=list(range(4)))
acc=accuracy_score(y_test2,y_pred)
metrics=[]
for k,name in enumerate(state_names):
    TP=cm[k,k]; FN=cm[k,:].sum()-TP; FP=cm[:,k].sum()-TP; TN=cm.sum()-TP-FN-FP
    sensitivity=TP/(TP+FN) if (TP+FN) else 0
    specificity=TN/(TN+FP) if (TN+FP) else 0
    metrics.append({'State (Activity)':name.title(),'Number of Samples':int(cm[k,:].sum()),'Sensitivity':round(sensitivity,3),'Specificity':round(specificity,3),'Overall Accuracy':round(acc,3)})
metrics_df=pd.DataFrame(metrics)
metrics_df.to_csv(PROJ/'data/processed/evaluation_metrics.csv',index=False)
pd.DataFrame(model.A_, index=state_names, columns=state_names).to_csv(PROJ/'data/processed/transition_matrix.csv')

# figures
plt.figure(figsize=(7,5)); im=plt.imshow(model.A_, aspect='auto'); plt.xticks(range(4),[s.title() for s in state_names]); plt.yticks(range(4),[s.title() for s in state_names]); plt.title('HMM Transition Matrix'); plt.xlabel('Next Activity'); plt.ylabel('Current Activity'); plt.colorbar(im)
for i in range(4):
    for j in range(4): plt.text(j,i,f'{model.A_[i,j]:.2f}',ha='center',va='center')
plt.tight_layout(); plt.savefig(PROJ/'figures/transition_matrix_heatmap.png', dpi=200); plt.close()
plt.figure(figsize=(8,5)); im=plt.imshow(cm, aspect='auto'); plt.xticks(range(4),[s.title() for s in state_names]); plt.yticks(range(4),[s.title() for s in state_names]); plt.title('Confusion Matrix on Unseen Data'); plt.xlabel('Predicted Activity'); plt.ylabel('True Activity'); plt.colorbar(im)
for i in range(4):
    for j in range(4): plt.text(j,i,str(cm[i,j]),ha='center',va='center')
plt.tight_layout(); plt.savefig(PROJ/'figures/confusion_matrix.png', dpi=200); plt.close()
plt.figure(figsize=(10,4)); plt.plot(y_test2, marker='o', label='True activity'); plt.plot(y_pred, marker='x', label='Predicted activity'); plt.yticks(range(4),[s.title() for s in state_names]); plt.title('Decoded Activity Sequence on Unseen Data'); plt.xlabel('Feature window'); plt.ylabel('Activity'); plt.legend(); plt.tight_layout(); plt.savefig(PROJ/'figures/decoded_sequence.png', dpi=200); plt.close()
# raw signal example
plt.figure(figsize=(10,4)); w=train_raw[train_raw.activity=='walking'].head(300); plt.plot(w.seconds_elapsed, np.sqrt(w.acc_x**2+w.acc_y**2+w.acc_z**2)); plt.title('Walking Accelerometer Magnitude'); plt.xlabel('Seconds'); plt.ylabel('Magnitude'); plt.tight_layout(); plt.savefig(PROJ/'figures/walking_accelerometer_magnitude.png', dpi=200); plt.close()

